In [1]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..","DTENV"))
Pkg.instantiate()
if size(LOAD_PATH,1) < 4
    push!(LOAD_PATH, joinpath(@__DIR__,"..","scripts"))
end

  Activating project at `c:\Users\Ivan\Desktop\Stuff4School\Thesis\CleanDTFE\DTENV`


4-element Vector{String}:
 "@"
 "@v#.#"
 "@stdlib"
 "c:\\Users\\Ivan\\Desktop\\Stuff4School\\Thesis\\CleanDTFE\\notebooks\\..\\scripts"

In [2]:
using TetGen
using StaticArrays
using JLD
using BenchmarkTools
using LinearAlgebra
using Plots

In [3]:
include("../scripts/TesselationCore.jl")
import .TesselationCore

BVH = TesselationCore.BVH
point3 = TesselationCore.point3

SVector{3, Float64} (alias for SArray{Tuple{3}, Float64, 1, 3})

In [4]:
import illustris_julia as il


basePath = "../../DTFE/Illustris3/output";

fields = ["SubhaloMass","SubhaloCM"];
subhalos = il.groupcat.loadSubhalos(basePath,135,fields)

positions = subhalos["SubhaloCM"]


3×121209 Matrix{Float32}:
   877.298    178.518    824.483  …  56394.3   64350.5  64728.4  61624.0
 26328.5    24557.1    26747.8        2440.16  31243.2  72960.7  66575.4
 18063.6    16858.7    17363.8       15078.2   39500.2  53679.3  32452.4

In [5]:
gap = 500
points = positions[:,1:gap:end]
ps = [point3(points[1,i], points[2,i], points[3,i]) for i in 1:size(points,2)]

bvh,tes,tets = TesselationCore.standardEstimator(ps,10)
print("done")

done

In [6]:
N = 1000

width = 75000

step = width/N


xs = bvh.bbox[1,1]:step:bvh.bbox[1,2]
ys = bvh.bbox[2,1]:step:bvh.bbox[2,2]

z = (bvh.bbox[3,2] + bvh.bbox[3,1])/2

dens = zeros(N,N)
print("Done")

Done

In [7]:
typeof(bvh)

Main.TesselationCore.Bvh.BVH

In [ ]:
# println(typeof(pts[1]))
# println(typeof(simps))
# println(typeof(bvh))


SVector{3, Float64}
Matrix{SVector{3, Float64}}
Main.TesselationCore.Bvh.BVH


In [12]:
for i in eachrow(pts)
    println(i)
end

[[324.5447292327881, 475.9657440185547, 37448.077239990234]]
[[699.5447292327881, 850.9657440185547, 37448.077239990234]]
[[1449.544729232788, 1600.9657440185547, 37448.077239990234]]
[[2199.544729232788, 2350.9657440185547, 37448.077239990234]]
[[3699.544729232788, 3850.9657440185547, 37448.077239990234]]
[[7449.544729232788, 7600.965744018555, 37448.077239990234]]


In [10]:
pts = [[xs[i],ys[i],z] for i in [5,10,20,30,50,100]]
simps = tes.points[tets]

# ids = [TesselationCore.findID(p, simps, bvh) for p in pts]

TesselationCore.DTFE(pts,bvh,tets,tes)

DimensionMismatch: DimensionMismatch: expected input array of length 3, got length 1

In [11]:
for (i,x) in pairs(xs)
    for (j,y) in pairs(ys)
        dens[i,j] = DTFE([x,y,z],bvh,tets,tes)

    end
end

UndefVarError: UndefVarError: `xs` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [12]:
med = median(dens)

UndefVarError: UndefVarError: `dens` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [13]:
Plots.heatmap(dens ./med,clim=(0,25))

UndefVarError: UndefVarError: `dens` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [12]:
N = 100

width = 75000

step = width/N


xs = bvh.bbox[1,1]:step:bvh.bbox[1,2]
ys = bvh.bbox[2,1]:step:bvh.bbox[2,2]

zs =  bvh.bbox[3,1]:step:bvh.bbox[3,2]

dens = zeros(N,N,N)
print("Done")

Done

In [13]:
DTFE([35000.,35000.,35000.],bvh,tets,tes)

2.1390745266731447e-12

In [14]:
for (i,x) in pairs(xs)
    for (j,y) in pairs(ys)
        for (k,z) in pairs(zs)
            dens[i,j,k] = DTFE([x,y,z],bvh,tets,tes)

        end

    end
end

In [15]:
using ColorSchemes
using GLMakie
lowColor  = get(ColorSchemes.acton,LinRange(0,1,256))[1]

fig = GLMakie.Figure(size = (1600,1600),backgroundcolor=lowColor)
ax = GLMakie.LScene(fig[1,1],scenekw=(show_axis=false,backgroundcolor=lowColor))
volplot = volume!(
    ax,dens ./median(dens),
    algorithm=:mip,
    colormap = :acton,
    colorrange = (.0,25),
    )

Volume{Tuple{Makie.EndPoints{Float32}, Makie.EndPoints{Float32}, Makie.EndPoints{Float32}, Array{Float32, 3}}}

In [16]:
fig

In [19]:
save("../Images/DTFE.png", fig)  